# Decision threshold modification by Hardt et al. - Adult data

This notebook contains an implementation of the post-processing fairness intervention introduced in [Equality of opportunity in supervised learning](https://dl.acm.org/doi/10.5555/3157382.3157469) by Hardt et al. (2016) as part of the IBM AIF360 fairness tool box github.com/IBM/AIF360.

The intervention method achieves equalised odds as follows. If only the decisions are available, they randomly choose either the original decision or fixed outcome in a way that ensures agreement across both protected groups. If a score function is available they choose between two carefully chosen thresholds with a particular probability to ensure agreement of true and false positive rates.

This method can be proven to be the optimal postprocessing algorithm for Equalised Odds, however the randomness introduced into decision making - which in particular could mean two identical individuals receive different outcomes - might clash with intuitive notions of fairness.

In [1]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from aif360.algorithms.postprocessing.eq_odds_postprocessing import (
    EqOddsPostprocessing,
)
from aif360.datasets import StandardDataset
from fairlearn.metrics import (
    equalized_odds_difference,
    false_negative_rate,
    false_positive_rate,
)
from helpers.metrics import accuracy
from helpers.plot import group_bar_plots

pip install 'aif360[AdversarialDebiasing]'


pip install 'aif360[AdversarialDebiasing]'


pip install 'aif360[inFairness]'


The baseline/FTU models were pickled with an older scikit-learn version, and this notebook pulls in libraries with newer, stricter validation than the versions the original CDEI project pinned in 2020. The cell below silences the resulting (harmless) version-compatibility warnings so the output stays focused on the fairness results.

In [2]:
import warnings

warnings.filterwarnings("ignore")

## Load data

We have committed preprocessed data to the repository for reproducibility and we load it here. Check out hte preprocessing notebook for details on how this data was obtained.

Location of artifacts (model and data)

In [3]:
artifacts_dir = Path("../../../artifacts")

In [4]:
data_dir = artifacts_dir / "data" / "adult"

train = pd.read_csv(data_dir / "processed" / "train-one-hot.csv")
val = pd.read_csv(data_dir / "processed" / "val-one-hot.csv")
test = pd.read_csv(data_dir / "processed" / "test-one-hot.csv")

In order to process data for our fairness intervention we need to define special dataset objects which are part of every intervention pipeline within the IBM AIF360 toolbox. These objects contain the original data as well as some useful further information, e.g., which feature is the protected attribute as well as which column corresponds to the label.

In [5]:
train_sds = StandardDataset(
    train,
    label_name="salary",
    favorable_classes=[1],
    protected_attribute_names=["sex"],
    privileged_classes=[[1]],
)
test_sds = StandardDataset(
    test,
    label_name="salary",
    favorable_classes=[1],
    protected_attribute_names=["sex"],
    privileged_classes=[[1]],
)
val_sds = StandardDataset(
    val,
    label_name="salary",
    favorable_classes=[1],
    protected_attribute_names=["sex"],
    privileged_classes=[[1]],
)
index = train_sds.feature_names.index("sex")

Define which binary value goes with the (un-)privileged group

In [6]:
privileged_groups = [{"sex": 1.0}]
unprivileged_groups = [{"sex": 0.0}]

## Load original model

For maximum reproducibility we can also load the baseline model from disk, but the code used to train can be found in the baseline model notebook.

In [7]:
bl_model = joblib.load(artifacts_dir / "models" / "finance" / "baseline.pkl")

Get prediction for validation and test data

In [8]:
bl_val_pred = bl_model.predict(val.drop("salary", axis=1))
val_sds_pred = val_sds.copy(deepcopy=True)
val_sds_pred.labels = bl_val_pred.reshape(-1, 1)

bl_test_pred = bl_model.predict(test.drop("salary", axis=1))
test_sds_pred = test_sds.copy(deepcopy=True)
test_sds_pred.labels = bl_test_pred.reshape(-1, 1)

## Equalised odds
Given the original unfair model we apply Hardt et al.'s intervention to achieve equalised odds. More precisely, we learn the mitigation procedure based on the true and predicted labels of the validation data. The learning does not need any parameter tuning. We then apply the learnt procedure to the predictions of the test data and analyse the outcomes for fairness and accuracy.

Note that this intervention method does not allow for a continuous trade-off between fairness and accuracy. Instead, the output is a single combination of accuracy and fairness for the corrected predictions.

In [9]:
# Learn parameters to equalize odds and apply to create a new dataset
eopp = EqOddsPostprocessing(
    privileged_groups=privileged_groups,
    unprivileged_groups=unprivileged_groups,
    seed=np.random.seed(),
)
eopp = eopp.fit(val_sds, val_sds_pred)

Apply intervention on test data.

In [10]:
test_sds_pred_tranf = eopp.predict(test_sds_pred)
test_scores = test_sds_pred_tranf.labels.flatten()

Analyse fairness and accuracy on the test data.

In [11]:
mask = test.sex == 1

male_fnr = false_negative_rate(test.salary[mask], test_scores[mask],)
female_fnr = false_negative_rate(test.salary[~mask], test_scores[~mask])
fnr_diff = np.abs(male_fnr - female_fnr)

male_fpr = false_positive_rate(test.salary[mask], test_scores[mask],)
female_fpr = false_positive_rate(test.salary[~mask], test_scores[~mask])
fpr_diff = np.abs(male_fpr - female_fpr)

print(f"Difference in false positive rates: {fpr_diff:.4f}")
print(f"Difference in false negative rates: {fnr_diff:.4f}")

Difference in false positive rates: 0.0043
Difference in false negative rates: 0.0060


In [12]:
acc = accuracy(test_scores.flatten(), test.salary)

female_acc = accuracy(
    test_scores.flatten()[test.sex == 0], test.salary[test.sex == 0],
)

male_acc = accuracy(
    test_scores.flatten()[test.sex == 1], test.salary[test.sex == 1],
)

print(f"Accuracy: {acc:.3f}")
print(f"Female accuracy: {female_acc:.3f}")
print(f"Male accuracy: {male_acc:.3f}")

print(f"Male false negative rate {male_fnr:.4f}")
print(f"Female false negative rate {female_fnr:.4f}")

print(f"Male false positive rate {male_fpr:.4f}")
print(f"Female false positive rate {female_fpr:.4f}")

Accuracy: 0.822
Female accuracy: 0.873
Male accuracy: 0.798
Male false negative rate 0.4531
Female false negative rate 0.4470
Male false positive rate 0.0899
Female false positive rate 0.0856


In [13]:
eod = equalized_odds_difference(
    test.salary, test_scores.flatten(), sensitive_features=test.sex,
)

print(f"Equalised odds: {eod:.4f}")

Equalised odds: 0.0060


We can visualise equalised odds using a bar plot. From this we see that the selection rates by outcome are almost identical for men and women.

In [14]:
eo_bar = group_bar_plots(
    test_scores,
    test.sex.map({0: "Female", 1: "Male"}),
    groups=test.salary,
    group_names=["Low earners", "High earners"],
    title="Proportion predicted high earners by sex and outcome",
    xlabel="Proportion predicted high earners",
    ylabel="Outcome",
)
eo_bar